In [139]:
import requests
import pandas as pd
import numpy as np
import math
import plotly.express as px
from dash import Dash, dcc, html
from jupyter_dash import JupyterDash
from joblib import Memory

In [ ]:
memory = Memory('../cache/features/', verbose=1)
@memory.cache
def get_FDA_data(endpoint, limit = 1000):
    """
    Fetches data from the FDA API for a specified endpoint and limit.

    Input:
    - endpoint: The specific FDA API endpoint to query (e.g., 'shortages', 'event').
    - limit: The maximum number of records to retrieve (default is 1000).
    - endpoints:  drug shortages: shortages.json
            drug adverse events: event.json

    Output: returns a list of records from the FDA API response.
    """
    #Store all records in a list
    all_records = []
    skip = 0 # Number of records to skip for pagination
    num_records = 0 # Total number of records available (to be updated after the first API call)

    while True:
        api_url = f"https://api.fda.gov/drug/{endpoint}?limit={limit}&skip={skip}"
        response = requests.get(api_url)
        
        if response.status_code == 200:
            try:
                data = response.json()  
                # Update total number of records available after the first API call
                if skip == 0:
                    # Number of records to be retrieved:
                    num_records = data.get('meta',{}).get('results', {}).get('total', 0)
                    print(f"Total records available: {num_records}")

                    # Number of queries needed to retrieve all records:
                    num_queries = math.ceil(num_records / limit)
                    print(f"Number of queries needed: {num_queries}")

                # Fetch the records and add to the list
                records = data.get('results', [])
                all_records.extend(records)
                print(f"Fetched records {skip + 1} to {skip + len(records)} of {num_records}")

            except ValueError:
                print("Error: Response content is not valid JSON.")

        else:
            print(f"Error: Failed to retrieve data. Status code: {response.status_code}")
            print("Error: Response content is not valid JSON.")

        skip += limit
        if skip >= num_records:
            break

    return all_records

In [141]:
df = pd.DataFrame(get_FDA_data('shortages.json', limit=1000))

"""
%%timeit
84.3 ms ± 2.36 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
"""

'\n%%timeit\n84.3 ms ± 2.36 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)\n'

In [ ]:
#Extracting relevant information from the 'openfda' column:
df['brand_name'] = df['openfda'].apply(lambda x: x.get('brand_name', [None])[0] if isinstance(x, dict) else None)
df['route'] = df['openfda'].apply(lambda x: (", ".join(y for y in x.get('route', [None]) if y) if isinstance(x, dict) else None))

#Extracting year from Initial posting date:
df['year'] = df['initial_posting_date'].dt.year

# Cleaning up the data:
df['initial_posting_date'] = pd.to_datetime(df['initial_posting_date'], errors='coerce')
df['update_date'] = pd.to_datetime(df['update_date'], errors='coerce')
df['discontinued_date'] = pd.to_datetime(df['discontinued_date'], errors='coerce')
df['therapeutic_category'] = df['therapeutic_category'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
df['availability'] = df['availability'].str.strip().replace('Limited Availabiltiy', 'Limited Availability')

#Dropping irrelevant columns:
df.drop(columns = ['package_ndc','contact_info','related_info','presentation','change_date', 'resolved_note', 'related_info_link','openfda'], inplace = True)


In [176]:
def update_shortage_reason(row):
    if pd.isnull(row['shortage_reason']):
        if row['availability'] == 'Available':
            row['shortage_reason'] = ' Available'
        elif row['availability'] == 'To Be Discontinued':
            row['shortage_reason'] = 'Not Applicable'
        elif row['availability'] == 'Resolved':
            row['shortage_reason'] = 'Not applicable'
        elif row['availability'] == 'Unavailable':
            row['shortage_reason'] = 'Unavailable'
        elif row['availability'] == 'Information pending':
            row['shortage_reason'] = 'Information pending'
    return row

# Instead of apply(update_availability):
mask_resolved = df['availability'].isna() & (df['status'] == 'Resolved')
mask_disc = df['availability'].isna() & (df['status'] == 'To Be Discontinued')
df.loc[mask_resolved, 'availability'] = 'Resolved'
df.loc[mask_disc, 'availability'] = 'To Be Discontinued'
df = df.apply(update_shortage_reason, axis=1)
df.isnull().sum()


update_type                0
initial_posting_date       0
generic_name               0
availability               0
update_date                0
therapeutic_category       0
dosage_form               11
company_name               0
shortage_reason            0
status                     0
discontinued_date       1192
brand_name               127
route                    122
dtype: int64

In [178]:
df.to_csv('../data/shortage_clean.csv', index=False)